# Evaluator Alignment Analysis

**Do the automated evaluators measure what a careful human would?**

This notebook *demonstrates* the analysis. It does not implement it — every statistic
below is computed by `evalforge.analytics.alignment`, which is unit-tested and is the
same code the dashboard and the release report call. A notebook that contained the only
implementation would be untestable and would drift from what the report prints.

---

> ## ⚠️ The annotations used here are SYNTHETIC
>
> They are generated by `scripts/simulate_annotations.py` so the pipeline is
> demonstrable offline. **They are not human judgements.** Every statistic below
> describes the annotation simulator, not human agreement, and must never be cited as
> evidence that the evaluators match human raters.
>
> Real numbers require `evalforge annotate` and real annotators.

---

## What this analysis answers

1. **What is the ceiling?** Human-versus-human agreement. No automated evaluator should
   be expected to agree with humans more than humans agree with each other, so an
   alignment figure reported without this number is uninterpretable.
2. **How well do the deterministic checks align?** These are the checks that gate releases.
3. **How well does the LLM judge align?** It scores the dimensions no exact rule reaches.
4. **Where does agreement break down?** By category, difficulty and conversation length.
5. **Are there systematic biases?** Verbosity, position, conciseness penalty, drift
   blindness, and reliability decay on long sessions.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evalforge.analytics.alignment import (
    build_alignment_report,
    cohens_kappa,
    pair_human_with_automated,
    pair_human_with_human,
)
from evalforge.analytics.statistics import interpret_kappa
from evalforge.config import load_config
from evalforge.storage.store import RunStore

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

config = load_config()
store = RunStore(config.paths.runs, config.storage.database_name)

runs = store.list_runs()
if not runs:
    raise SystemExit("No stored runs. Run `evalforge demo` first.")

pd.DataFrame(
    [
        {
            "run_id": r.run_id,
            "label": r.label,
            "sessions": r.session_count,
            "decision": r.release_decision.value,
            "pass_rate": round(r.metrics.get("pass_rate", 0.0), 4),
        }
        for r in runs
    ]
)

## 1. Load a run and its annotations

Only **blind** annotations are eligible. An annotator who has already seen the automated
verdict is not an independent rater, and pooling them would inflate every agreement
number. `get_annotations` filters to blind by default.

In [ ]:
run = next((r for r in runs if r.label == "baseline"), runs[0])
sessions = store.get_sessions(run.run_id)
annotations = store.get_annotations(run.run_id, blind_only=True)

synthetic = sum(1 for a in store.get_annotations(run.run_id, blind_only=False)
                if a.metadata.get("synthetic"))

print(f"run          : {run.run_id} ({run.label})")
print(f"sessions     : {len(sessions)}")
print(f"annotations  : {len(annotations)} blind")
print(f"annotators   : {len({a.annotator_id for a in annotations})}")
if synthetic:
    print(f"\n*** {synthetic} of these annotations are SYNTHETIC. "
          "Statistics below describe the simulator, not human agreement. ***")

if not annotations:
    print("\nNo annotations. Generate demonstration ones with:")
    print("    python scripts/simulate_annotations.py --all-runs")

## 2. The full alignment report

One call computes everything: agreement statistics, confusion matrices, subgroup
breakdowns, bias findings and disagreement examples.

In [ ]:
verbosity = {}
for session in sessions:
    try:
        trace = store.get_trace(run.run_id, session.session_id)
        verbosity[session.session_id] = sum(len(t.assistant_message) for t in trace.turns)
    except Exception:
        continue

report = build_alignment_report(run.run_id, annotations, sessions, verbosity)

print(f"annotations      : {report.annotation_count}")
print(f"annotators       : {report.annotator_count}")
print(f"doubly annotated : {report.doubly_annotated}")
print(f"statistics       : {len(report.statistics)}")
print(f"bias findings    : {len(report.bias_findings)}")

## 3. The ceiling: human versus human

Cohen's kappa corrects for chance agreement, which matters enormously here. With an 85%
pass rate, two raters who both always say "pass" show 85% *raw* agreement and know
nothing — kappa is near zero, and kappa is the honest number.

In [ ]:
human_pairs = pair_human_with_human(annotations)

rows = []
for pair in human_pairs:
    kappa = cohens_kappa(pair.a_pass, pair.b_pass)
    raw = sum(1 for x, y in zip(pair.a_pass, pair.b_pass) if x == y) / len(pair)
    rows.append({
        "rater A": pair.rater_a,
        "rater B": pair.rater_b,
        "n": len(pair),
        "raw agreement": round(raw, 4),
        "cohens kappa": round(kappa, 4),
        "reading": interpret_kappa(kappa),
    })

ceiling = pd.DataFrame(rows)
display(ceiling)

if rows:
    print(f"\nCeiling for automated agreement: kappa = {rows[0]['cohens kappa']}")
    print("An automated evaluator scoring meaningfully above this would be suspicious,")
    print("not impressive — it would mean it agrees with humans more than they agree.")

## 4. Human versus each automated source

Three comparisons: the deterministic checks (which gate releases), the LLM judge (which
does not), and the aggregate verdict.

In [ ]:
rows = []
for source in ("deterministic", "judge", "aggregate"):
    pair = pair_human_with_automated(annotations, sessions, source)
    if len(pair) < 5:
        continue
    kappa = cohens_kappa(pair.a_pass, pair.b_pass)
    raw = sum(1 for x, y in zip(pair.a_pass, pair.b_pass) if x == y) / len(pair)
    rows.append({
        "automated source": source,
        "n": len(pair),
        "raw agreement": round(raw, 4),
        "cohens kappa": round(kappa, 4),
        "reading": interpret_kappa(kappa),
    })

display(pd.DataFrame(rows))

print("\nReminder: the deterministic row is the one that matters for release decisions.")
print("Judge scores are reported alongside and never gate a release (ADR-002).")

## 5. All computed statistics

Including weighted kappa on the ordinal rubric dimensions — on a 1..5 scale a 4-vs-5
disagreement is not the same as 1-vs-5, so quadratic weighting penalises distant
disagreements more.

In [ ]:
stats = pd.DataFrame([
    {
        "statistic": s.name,
        "rater A": s.rater_a,
        "rater B": s.rater_b,
        "dimension": s.dimension or "overall",
        "value": s.value,
        "n": s.n,
        "reading": s.interpretation,
    }
    for s in report.statistics
])
display(stats.sort_values(["statistic", "rater A"]).reset_index(drop=True))

## 6. Confusion matrices

The dangerous cell when B is automated is **A failed / B passed**: a false pass ships a
broken agent.

In [ ]:
for name, matrix in report.confusion_matrices.items():
    print(f"\n{name.replace('_', ' ')}")
    display(pd.DataFrame([
        {"": "A passed", "B passed": matrix["true_pass"], "B failed": matrix["false_fail"]},
        {"": "A failed", "B passed": matrix["false_pass"], "B failed": matrix["true_fail"]},
    ]).set_index(""))

    total = sum(matrix.values())
    if total:
        fp = matrix["false_pass"] / total
        fn = matrix["false_fail"] / total
        print(f"  false-pass rate {fp:.1%}  |  false-fail rate {fn:.1%}")

## 7. Where agreement breaks down

The hypothesis worth testing: evaluator reliability decays as conversation length grows.
Longer sessions are harder for a human to hold in mind *and* give the automated checks
more surface to be wrong on.

In [ ]:
by_length = pd.DataFrame(
    [{"turns": int(k), "agreement": v} for k, v in report.agreement_by_length.items()]
).sort_values("turns")
display(by_length)

# Rendered as text rather than a chart: matplotlib is not a project dependency, and the
# dashboard already plots this. The trend is the part that matters.
if len(by_length) > 1:
    for _, row in by_length.iterrows():
        turns = int(row["turns"])
        agreement = float(row["agreement"])
        bar = "#" * int(round(agreement * 40))
        print("{:>3} turns | {:<40} {:.3f}".format(turns, bar, agreement))

    first_agreement = float(by_length.iloc[0]["agreement"])
    last_agreement = float(by_length.iloc[-1]["agreement"])
    drop = first_agreement - last_agreement

    print("\nShortest sessions: {:.3f}".format(first_agreement))
    print("Longest sessions : {:.3f}".format(last_agreement))
    print("Change across the length sweep: {:+.3f}".format(-drop))

    if drop > 0.05:
        print("-> Supports the hypothesis that evaluator reliability decays with length.")
    elif drop < -0.05:
        print("-> Agreement improves with length, which is worth investigating.")
    else:
        print("-> No material change across lengths in this sample.")

print("\nBy scenario category:")
display(pd.DataFrame(
    [{"category": k, "agreement": v} for k, v in report.agreement_by_failure_type.items()]
).sort_values("agreement"))

print("\nBy difficulty:")
display(pd.DataFrame(
    [{"difficulty": k, "agreement": v} for k, v in report.agreement_by_difficulty.items()]
))


## 8. Bias analysis

Five biases are checked. The two most important:

- **Verbosity bias** — the classic judge failure mode. Measured as the *residual*
  correlation between output length and automated score after subtracting the human
  correlation with length. A positive residual means the automated score rewards length
  more than humans do.
- **Subtle goal drift** — the category both humans and automated checks are expected to
  be weakest on, since every individual turn looks helpful.

In [ ]:
for finding in report.bias_findings:
    print(f"\n### {finding['bias'].replace('_', ' ').title()}")
    for key, value in finding.items():
        if key in {"bias", "interpretation"}:
            continue
        print(f"  {key}: {value}")
    if finding.get("interpretation"):
        print(f"  -> {finding['interpretation']}")

## 9. Disagreements worth adjudicating

Sorted by score gap. Where a human and the automated verdict diverge, one of them is
wrong — and because every automated verdict carries trace evidence, the disagreement is
adjudicable rather than a matter of opinion.

In [ ]:
disagreements = pd.DataFrame(report.disagreement_examples)
if disagreements.empty:
    print("Human and automated verdicts agreed on every annotated session.")
else:
    display(disagreements[[
        "scenario_id", "category", "turn_count",
        "human_pass", "automated_pass", "human_score", "automated_score", "gap",
    ]])

    worst = report.disagreement_examples[0]
    print(f"\nLargest gap: {worst['scenario_id']} ({worst['category']})")
    print(f"  human: {'pass' if worst['human_pass'] else 'fail'} "
          f"({worst['human_score']}) | automated: "
          f"{'pass' if worst['automated_pass'] else 'fail'} ({worst['automated_score']})")
    print(f"  automated failure categories: {worst['automated_categories']}")
    print(f"\n  Inspect it: evalforge inspect --run-id {run.run_id} "
          f"--scenario-id {worst['scenario_id']}")

## 10. Limitations of this analysis

Computed and reported by the analysis itself, not asserted by hand.

In [ ]:
for item in report.limitations:
    print(f"- {item}")

print("\nStanding limitations regardless of sample size:")
print("- The annotations here are synthetic; these numbers describe the simulator.")
print("- Two annotators is the minimum for a kappa, well below a serious study.")
print("- Annotators were not calibrated against each other beforehand.")
print("- The rubric has not been independently validated for inter-rater reliability.")
print("\nSee docs/LIMITATIONS.md for the full account.")

---

## Conclusion

The analysis pipeline works end to end: agreement statistics, chance correction, ordinal
weighting, subgroup breakdowns, bias detection and adjudicable disagreements.

**What it does not yet show** is whether EvalForge's evaluators agree with *people*,
because the annotations backing it are simulated. Replacing them is the highest-value
next step for the project, and the interface to do it already exists:

```bash
evalforge annotate
```